# 🥈 Silver Layer – Cleansing & Transformation
**Chicago Crime Dataset – ETL Workflow Automation**

Reads from Bronze Delta table, applies data quality rules, type casting, deduplication,
and writes a clean, enriched dataset to the Silver layer.

In [0]:
# ─────────────────────────────────────────────
# 1. PARAMETERS
# ─────────────────────────────────────────────
dbutils.widgets.text("bronze_container",  "bronze",                   "Bronze Container")
dbutils.widgets.text("bronze_path",       "chicago_crime/",           "Bronze Path")
dbutils.widgets.text("silver_container",  "silver",                   "Silver Container")
dbutils.widgets.text("silver_path",       "chicago_crime/",           "Silver Path")
dbutils.widgets.text("storage_account",   "etlworkflowautomation",   "ADLS Storage Account")
dbutils.widgets.text("run_date",          "",                         "Run Date (YYYY-MM-DD)")
dbutils.widgets.text("pipeline_run_id",   "manual",                   "ADF Pipeline Run ID")
dbutils.widgets.text("dq_fail_threshold", "0.05",                     "Max bad-row fraction before abort")

BRONZE_CONTAINER   = dbutils.widgets.get("bronze_container")
BRONZE_PATH        = dbutils.widgets.get("bronze_path")
SILVER_CONTAINER   = dbutils.widgets.get("silver_container")
SILVER_PATH        = dbutils.widgets.get("silver_path")
STORAGE_ACCOUNT    = dbutils.widgets.get("storage_account")
RUN_DATE           = dbutils.widgets.get("run_date")
PIPELINE_RUN_ID    = dbutils.widgets.get("pipeline_run_id")
DQ_FAIL_THRESHOLD  = float(dbutils.widgets.get("dq_fail_threshold"))

In [0]:
# ─────────────────────────────────────────────
# 2. IMPORTS
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, upper, trim, when, isnan, isnull,
    year, month, dayofweek, hour, lit, current_timestamp,
    count, sum as spark_sum
)
from pyspark.sql.types import BooleanType
from datetime import datetime, date
import traceback

spark = SparkSession.builder.appName("Silver_Chicago_Crime_Transform").getOrCreate()

SECRET_SCOPE = "etlworkflowautomation"
SECRET_KEY   = "*******************************"  
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SECRET_KEY
)

RUN_DATE   = RUN_DATE if RUN_DATE else str(date.today())
START_TIME = datetime.now()
print(f"Run date : {RUN_DATE}  |  DQ threshold : {DQ_FAIL_THRESHOLD}")

Run date : 2026-05-20 | DQ threshold : 0.05

In [0]:
# ─────────────────────────────────────────────
# 3. READ BRONZE (today's partition only)
# ─────────────────────────────────────────────
bronze_uri = f"wasbs://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{BRONZE_PATH}"

try:
    df_bronze = (
        spark.read
        .format("delta")
        .load(bronze_uri)
    )
    bronze_count = df_bronze.count()
    print(f"Bronze rows loaded for {RUN_DATE}: {bronze_count:,}")
except Exception as e:
    print(f"Bronze read failed: {e}")
    dbutils.notebook.exit(f"FAILED|0|{str(e)}")

Bronze rows loaded for 2026-05-20: 8,554,479

In [0]:
# ─────────────────────────────────────────────
# 4. DATA QUALITY CHECKS
# ─────────────────────────────────────────────
dq_results = {}

# 4a. Null critical fields
null_id          = df_bronze.filter(col("id").isNull()).count()
null_case_num    = df_bronze.filter(col("case_number").isNull()).count()
null_date        = df_bronze.filter(col("date").isNull()).count()
null_ptype       = df_bronze.filter(col("primary_type").isNull()).count()
null_lat         = df_bronze.filter(col("latitude").isNull()).count()
null_lon         = df_bronze.filter(col("longitude").isNull()).count()

dq_results["null_id"]           = null_id
dq_results["null_case_number"]  = null_case_num
dq_results["null_date"]         = null_date
dq_results["null_primary_type"] = null_ptype
dq_results["null_latitude"]     = null_lat
dq_results["null_longitude"]    = null_lon

# 4b. Duplicate id
dup_count = bronze_count - df_bronze.dropDuplicates(["id"]).count()
dq_results["duplicate_id"] = dup_count

# 4c. Coordinate range sanity (Chicago bounding box)
out_of_bounds = df_bronze.filter(
    col("latitude").isNotNull() &
    ((col("latitude")  < 41.6) | (col("latitude")  > 42.1) |
     (col("longitude") < -87.9) | (col("longitude") > -87.5))
).count()
dq_results["coord_out_of_bounds"] = out_of_bounds

total_bad    = null_id + null_date + dup_count
bad_fraction = total_bad / bronze_count if bronze_count > 0 else 0

print("\n── Data Quality Report ──────────────────")
for k, v in dq_results.items():
    print(f"  {k:<30}: {v:>8,}")
print(f"  {'bad_row_fraction':<30}: {bad_fraction:.4f}")
print("─────────────────────────────────────────")

if bad_fraction > DQ_FAIL_THRESHOLD:
    msg = f"DQ check FAILED: bad_fraction={bad_fraction:.4f} > threshold={DQ_FAIL_THRESHOLD}"
    print(f"{msg}")
    dbutils.notebook.exit(f"FAILED|{bronze_count}|{msg}")
else:
    print("DQ checks passed.")

── Data Quality Report ──────────────────
 null_id : 0
 null_case_number : 0
 null_date : 0
 null_primary_type : 0
 null_latitude : 96,604
 null_longitude : 96,604
 duplicate_id : 0
 coord_out_of_bounds : 27,965
 bad_row_fraction : 0.0000
─────────────────────────────────────────
DQ checks passed.

In [0]:
# ─────────────────────────────────────────────
# 5. CLEANSING & TYPE CASTING
# ─────────────────────────────────────────────
DATE_FORMAT = "MM/dd/yyyy hh:mm:ss a"   # Chicago crime CSV date format

df_cleaned = (
    df_bronze
    # Deduplication – keep latest ingestion per unique_key
    .dropDuplicates(["id"])
    # Drop rows with critical nulls
    .filter(col("id").isNotNull() & col("date").isNotNull())
    # Parse date string → proper timestamp
    .withColumn("crime_timestamp",  to_timestamp(col("date"), DATE_FORMAT))
    # Standardise strings
    .withColumn("primary_type",     upper(trim(col("primary_type"))))
    .withColumn("description",      upper(trim(col("description"))))
    .withColumn("location_description", upper(trim(col("location_description"))))
    .withColumn("block",            upper(trim(col("block"))))
    .withColumn("iucr",             trim(col("iucr")))
    # Fill nulls for non-critical fields
    .fillna({"location_description": "UNKNOWN", "description": "UNKNOWN"})
)

In [0]:
# ─────────────────────────────────────────────
# 6. FEATURE ENGINEERING (derived columns)
# ─────────────────────────────────────────────
df_silver = (
    df_cleaned
    .withColumn("crime_year",       year(col("crime_timestamp")))
    .withColumn("crime_month",      month(col("crime_timestamp")))
    .withColumn("crime_hour",       hour(col("crime_timestamp")))
    .withColumn("crime_dayofweek",  dayofweek(col("crime_timestamp")))
    # Convenience flag
    .withColumn("is_night",
        when((col("crime_hour") >= 20) | (col("crime_hour") < 6), True).otherwise(False)
    )
    .withColumn("is_weekend",
        when(col("crime_dayofweek").isin(1, 7), True).otherwise(False)
    )
    # Audit
    .withColumn("_silver_processed_at", current_timestamp())
    .withColumn("_pipeline_run_id",     lit(PIPELINE_RUN_ID))
    .withColumn("_run_date",            lit(RUN_DATE))
    # Drop raw date string and Bronze audit columns
    .drop("date", "_ingested_at", "_source_file")
)

silver_count = df_silver.count()
print(f"Silver rows after cleansing: {silver_count:,}  (dropped {bronze_count - silver_count:,} rows)")
df_silver.printSchema()

Silver rows after cleansing: 8,554,479 (dropped 0 rows)
root
-- id: long (nullable = true)
-- case_number: string (nullable = true)
-- block: string (nullable = true)
-- iucr: string (nullable = true)
-- primary_type: string (nullable = true)
-- description: string (nullable = false)
-- location_description: string (nullable = false)
-- arrest: boolean (nullable = true)
-- domestic: boolean (nullable = true)
-- beat: integer (nullable = true)
-- district: integer (nullable = true)
-- ward: integer (nullable = true)
-- community_area: integer (nullable = true)
-- fbi_code: string (nullable = true)
-- x_coordinate: double (nullable = true)
-- y_coordinate: double (nullable = true)
-- year: integer (nullable = true)
-- updated_on: string (nullable = true)
-- latitude: double (nullable = true)
-- longitude: double (nullable = true)
-- location: string (nullable = true)
-- _pipeline_run_id: string (nullable = false)
-- _run_date: string (nullable = false)
-- crime_timestamp: timestamp (nullable = true)
-- crime_year: integer (nullable = true)
-- crime_month: integer (nullable = true)
-- crime_hour: integer (nullable = true)
-- crime_dayofweek: integer (nullable = true)
-- is_night: boolean (nullable = false)
-- is_weekend: boolean (nullable = false)
-- _silver_processed_at: timestamp (nullable = false)

In [0]:
# ─────────────────────────────────────────────
# 7. WRITE TO SILVER LAYER (Delta, merge logic)
# ─────────────────────────────────────────────
from delta.tables import DeltaTable

silver_uri = f"wasbs://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{SILVER_PATH}"

try:
    # If table exists, UPSERT to avoid duplicates across reruns
    if DeltaTable.isDeltaTable(spark, silver_uri):
        dt = DeltaTable.forPath(spark, silver_uri)
        (
            dt.alias("existing")
            .merge(
                df_silver.alias("incoming"),
                "existing.id = incoming.id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print("Silver UPSERT complete (MERGE strategy).")
    else:
        (
            df_silver
            .write
            .format("delta")
            .mode("overwrite")
            .partitionBy("crime_year", "crime_month")
            .save(silver_uri)
        )
        print("Silver table created (initial load).")
except Exception as e:
    print(f"Silver write failed: {e}")
    traceback.print_exc()
    dbutils.notebook.exit(f"FAILED|{silver_count}|{str(e)}")

Silver UPSERT complete (MERGE strategy).

In [0]:
# ─────────────────────────────────────────────
# 8. REGISTER TABLE
# ─────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS chicago_crime")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS chicago_crime.silver_crimes
    SELECT * FROM delta.`{silver_uri}`
""")
print("Delta table 'chicago_crime.silver_crimes' registered.")

Delta table 'chicago_crime.silver_crimes' registered.

In [0]:
# ─────────────────────────────────────────────
# 9. EXIT
# ─────────────────────────────────────────────
duration_sec = (datetime.now() - START_TIME).total_seconds()
print(f"Duration : {duration_sec:.1f}s  |  Rows written : {silver_count:,}")
dbutils.notebook.exit(f"SUCCESS|{silver_count}|{duration_sec:.1f}")